### GPU 확인

In [13]:
import ultralytics
ultralytics.checks()

Ultralytics 8.3.141 🚀 Python-3.12.3 torch-2.7.0+cu126 CUDA:0 (NVIDIA GeForce RTX 2060, 5740MiB)
Setup complete ✅ (12 CPUs, 15.4 GB RAM, 409.8/467.3 GB disk)


In [14]:
import torch

def check_cuda_availability():
    if torch.cuda.is_available():
        print("CUDA is available")
        device = torch.device("cuda")
    else:
        print("CUDA is NOT available")
        device = torch.device("cpu")

    return device

In [15]:
import platform

if platform.system() == "Darwin":
    print("Your system is  Linux")
    device = check_cuda_availability()
else:
    print("Your system is ", platform.system())
    device = check_cuda_availability()

print("Using device: ", device)

Your system is  Linux
CUDA is available
Using device:  cuda


### 라이브러리 선언

In [16]:
import cv2
import mediapipe as mp
from ultralytics import YOLO
import os
import math
import warnings
import logging
from tqdm import tqdm

### 초기화

In [17]:
mp_pose = mp.solutions.pose
yolo_model = YOLO('yolov8n.pt', verbose=False)

### 포즈 관절점 추출

In [18]:
def get_pose_keypoint(img):
    with mp_pose.Pose() as pose:
        results = pose.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

        if results.pose_landmarks:
            keypoints = []
            for landmark in results.pose_landmarks.landmark:
                keypoints.extend([landmark.x, landmark.y, landmark.z])
            return keypoints
        else:
            return [0.0] * 99

### 객체 감지 (가방, 물체)

In [19]:
def get_objects(img):
    """
    YOLO로 큰 객체만 감지 (가방, 쇼핑백)
    - 작은 물건들은 제외
    - 신뢰도 높은 것만 선택
    """
    
    results = yolo_model(img, verbose=False, device=device)
    objects = []
    
    for r in results:
        if r.boxes is not None:
            for box in r.boxes:
                cls = int(box.cls)      # 클래스 번호
                conf = float(box.conf)  # 신뢰도
                
                # 가방류만 선택 (클래스 24, 26, 27)
                if cls in [24, 26, 27] and conf > 0.6:
                    x1, y1, x2, y2 = box.xyxy[0].tolist()  # 박스 좌표
                    
                    # 박스가 너무 작으면 제외
                    box_area = (x2-x1) * (y2-y1)
                    img_area = img.shape[0] * img.shape[1]
                    
                    if box_area > img_area * 0.01:  # 이미지의 1% 이상 크기
                        objects.append([x1, y1, x2, y2, conf, cls])
    
    return objects

### 손목과 가장 가까운 객체 찾기

In [20]:
def find_nearest_objects(pose_keypoints, objects, img_width, img_height):
    """
    손목과 가장 가까운 객체들 찾기
    """
    if len(pose_keypoints) < 99:  # 관절점이 없으면
        return []
    
    # 손목 좌표 추출 (MediaPipe 인덱스: 15=왼쪽손목, 16=오른쪽손목)
    left_wrist_x = pose_keypoints[15*3] * img_width     # 15번 점의 x
    left_wrist_y = pose_keypoints[15*3 + 1] * img_height # 15번 점의 y
    right_wrist_x = pose_keypoints[16*3] * img_width     # 16번 점의 x  
    right_wrist_y = pose_keypoints[16*3 + 1] * img_height # 16번 점의 y
    
    near_objects = []
    
    for obj in objects:
        x1, y1, x2, y2, conf, cls = obj
        
        # 객체 중심점 계산
        obj_center_x = (x1 + x2) / 2
        obj_center_y = (y1 + y2) / 2
        obj_width = x2 - x1
        obj_height = y2 - y1
        
        # 각 손목과 객체 중심 사이의 거리 계산
        left_dist = math.sqrt((left_wrist_x - obj_center_x)**2 + (left_wrist_y - obj_center_y)**2)
        right_dist = math.sqrt((right_wrist_x - obj_center_x)**2 + (right_wrist_y - obj_center_y)**2)
        
        # 더 가까운 거리 선택
        min_dist = min(left_dist, right_dist)
        
        # 손목과 가까운 객체만 선택 (이미지 대각선의 30% 이내)
        diagonal = math.sqrt(img_width**2 + img_height**2)
        if min_dist < diagonal * 0.3:
            # 정규화된 값들로 저장
            near_objects.append([
                obj_center_x/img_width,  # 중심 x (정규화)
                obj_center_y/img_height, # 중심 y (정규화)
                obj_width/img_width,     # 너비 (정규화)
                obj_height/img_height,   # 높이 (정규화)
                conf,                    # 신뢰도
                cls,                     # 클래스
                min_dist/diagonal        # 거리 (정규화)
            ])
    
    # 거리순 정렬하여 가장 가까운 3개만 반환
    near_objects.sort(key=lambda x: x[6])  # 거리로 정렬
    return near_objects[:2]

In [21]:
def extract_features(img_path):
    """포즈 + 손목근처 객체 특징 추출"""
    img = cv2.imread(img_path)
    if img is None:
        return None
        
    img_height, img_width = img.shape[:2]
    
    # 1. 포즈 키포인트 (99개)
    pose_features = get_pose_keypoint(img)
    
    # 2. 객체 감지
    objects = get_objects(img)
    
    # 3. 손목과 가까운 객체 (최대 2개)
    near_objects = find_nearest_objects(pose_features, objects, img_width, img_height)
    
    # 4. 특징 통합
    all_features = []
    all_features.extend(pose_features)  # 99개
    
    # 객체 정보 (2개 * 7 = 14개, 부족하면 0으로 패딩)
    for i in range(2):
        if i < len(near_objects):
            all_features.extend(near_objects[i])
        else:
            all_features.extend([0.0] * 7)
    
    return all_features  # 총 99 + 14 = 113개 특징

### txt 파일로 저장

In [22]:
def save_to_txt(features, output_path, label):
    if features is None:
        return False
        
    with open(output_path, 'w') as f:
         # 라벨 먼저 (0=normal, 1=abandon)
        f.write(f"{label}")
        for feature in features:
            f.write(f" {feature:.6f}")
        f.write("\n")
    
    return True

### 이미지 폴더 일괄 처리

In [23]:
def process_folder(image_folder, features_folder, label):
    """이미지 폴더 → 특징 폴더로 분리 처리"""
    if not os.path.exists(image_folder):
        print(f"❌ 이미지 폴더가 없습니다: {image_folder}")
        return 0
    
    # 특징 폴더 생성
    os.makedirs(features_folder, exist_ok=True)
    
    # 이미지 파일 리스트
    image_files = [f for f in os.listdir(image_folder) 
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    
    if len(image_files) == 0:
        print(f"📁 {os.path.basename(image_folder)}: 이미지 없음")
        return 0
    
    folder_name = os.path.basename(os.path.dirname(image_folder))  # abandon or normal
    split_name = os.path.basename(os.path.dirname(os.path.dirname(image_folder)))  # train or val
    label_name = "ABANDON" if label == 3 else "NORMAL"
    
    print(f"📁 {split_name}/{folder_name} ({label_name}): {len(image_files)}개")
    print(f"   📂 이미지: {image_folder}")  
    print(f"   💾 특징: {features_folder}")
    
    success_count = 0
    
    # tqdm 진행바
    for img_file in tqdm(image_files, 
                        desc=f"{split_name}-{folder_name[:6]} [GPU]", 
                        unit="img",
                        colour="red" if label == 1 else "green"):
        
        img_path = os.path.join(image_folder, img_file)
        
        # 특징 추출
        features = extract_features(img_path)
        
        if features is not None:
            # txt 파일은 features 폴더에 저장
            txt_filename = os.path.splitext(img_file)[0] + '.txt'
            txt_path = os.path.join(features_folder, txt_filename)
            
            if save_to_txt(features, txt_path, label):
                success_count += 1
    
    print(f"   ✅ 완료: {success_count}/{len(image_files)}개\n")
    return success_count

### txt 생성

In [ ]:
def main():
    
    base_path = "../data/frames"
    total_processed = 0
    
    # 🔴 TRAIN 데이터 처리
    print("🔴 TRAIN 데이터 처리")
    print("-" * 30)
    
    # Train abandon
    train_abandon_images = f"{base_path}/abandon/train/images"
    train_abandon_features = f"{base_path}/abandon/train/features"
    total_processed += process_folder(train_abandon_images, train_abandon_features, label=3)
    
    # Train normal  
    train_normal_images = f"{base_path}/abandon/normal/train/images"
    train_normal_features = f"{base_path}/abandon/normal/train/features"
    total_processed += process_folder(train_normal_images, train_normal_features, label=0)
    
    # 🔵 VAL 데이터 처리
    print("🔵 VAL 데이터 처리") 
    print("-" * 30)
    
    # Val abandon
    val_abandon_images = f"{base_path}/abandon/val/images"
    val_abandon_features = f"{base_path}/abandon/val/features"
    total_processed += process_folder(val_abandon_images, val_abandon_features, label=3)
    
    # Val normal
    val_normal_images = f"{base_path}/abandon/normal/val/images"
    val_normal_features = f"{base_path}/abandon/normal/val/features"
    total_processed += process_folder(val_normal_images, val_normal_features, label=0)
    
    print("🎉 전체 처리 완료!")
    print(f"📊 총 처리된 파일: {total_processed:,}개")


if __name__ == "__main__":
    main()

🔴 TRAIN 데이터 처리
------------------------------
📁 abandon/train (ABANDON): 15185개
   📂 이미지: ../data/frames/abandon/train/images
   💾 특징: ../data/frames/abandon/train/features


abandon-train [GPU]:   0%|          | 0/15185 [00:00<?, ?img/s]

I0000 00:00:1749201756.499351 1959841 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1749201756.512430 1966168 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 24.2.8-1ubuntu1~24.04.1), renderer: Mesa Intel(R) UHD Graphics (CML GT2)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1749201756.724387 1966162 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1749201756.845169 1966154 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1749201756.959733 1966155 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
